# Cálculo Numérico — Lista de Exercícios
**Sistemas lineares e implementação numérica**

Aluno(a): Felipe Marinho de Figueiredo  &nbsp;&nbsp; RA: 185607

Cada exercício traz os passos da resolução manual (em Markdown/LaTeX) e, quando pedido, o código Python comentado, os resultados e uma breve interpretação.

# Parte 1 — Conceitos e resolução manual

## Exercício 1 — Sistemas lineares na forma matricial $A\mathbf{x}=\mathbf{b}$

Cada linha de $A$ recebe os coeficientes de uma equação, na ordem das incógnitas.

**a)**
$$A=\begin{bmatrix}2&1\\1&-3\end{bmatrix},\qquad \mathbf{x}=\begin{bmatrix}x\\y\end{bmatrix},\qquad \mathbf{b}=\begin{bmatrix}7\\-5\end{bmatrix}$$

**b)**
$$A=\begin{bmatrix}1&2&-1\\3&-1&2\\2&1&1\end{bmatrix},\qquad \mathbf{x}=\begin{bmatrix}x\\y\\z\end{bmatrix},\qquad \mathbf{b}=\begin{bmatrix}2\\9\\7\end{bmatrix}$$

## Exercício 2 — Eliminação de Gauss

Sistema:
$$\begin{cases}2x+y-z=8\\-3x-y+2z=-11\\-2x+y+2z=-3\end{cases}$$

Matriz aumentada inicial:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\-3&-1&2&-11\\-2&1&2&-3\end{array}\right]$$

**Passo 1 — zerar a coluna 1 abaixo do pivô (2).** Multiplicadores: $m_{21}=-3/2$ e $m_{31}=-2/2=-1$.

- $L_2\leftarrow L_2+\tfrac32 L_1$
- $L_3\leftarrow L_3+L_1$

$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&0{,}5&0{,}5&1\\0&2&1&5\end{array}\right]$$

**Passo 2 — zerar a coluna 2 abaixo do pivô (0,5).** Multiplicador: $m_{32}=2/0{,}5=4$.

- $L_3\leftarrow L_3-4L_2$

$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&0{,}5&0{,}5&1\\0&0&-1&1\end{array}\right]$$

**Substituição regressiva:**

- $-z=1 \Rightarrow z=-1$
- $0{,}5y+0{,}5(-1)=1 \Rightarrow y=3$
- $2x+3-(-1)=8 \Rightarrow x=2$

$$\boxed{(x,y,z)=(2,\,3,\,-1)}$$

Verificação na 2ª equação: $-3(2)-3+2(-1)=-11$ ✓

# Parte 2 — Propriedades e interpretação

## Exercício 3 — Número de soluções

**a)** $\left[\begin{array}{cc|c}1&1&3\\2&2&6\end{array}\right]$. Com $L_2\leftarrow L_2-2L_1$ obtemos $\left[\begin{array}{cc|c}1&1&3\\0&0&0\end{array}\right]$.

A segunda equação é redundante: sobra 1 equação para 2 incógnitas, logo há **infinitas soluções** ($x=3-y$, com $y$ livre). Aqui $\text{posto}(A)=\text{posto}([A|b])=1<2$.

**b)** $\left[\begin{array}{cc|c}1&1&3\\2&2&7\end{array}\right]$. Com $L_2\leftarrow L_2-2L_1$ obtemos $\left[\begin{array}{cc|c}1&1&3\\0&0&1\end{array}\right]$, ou seja, $0=1$.

Contradição: **não há solução**. Aqui $\text{posto}(A)=1<\text{posto}([A|b])=2$.

**c)** $\left[\begin{array}{cc|c}1&1&3\\2&-1&0\end{array}\right]$. Com $L_2\leftarrow L_2-2L_1$ obtemos $\left[\begin{array}{cc|c}1&1&3\\0&-3&-6\end{array}\right]$.

Então $y=2$ e $x=1$: **solução única** $(1,2)$. Aqui $\text{posto}(A)=\text{posto}([A|b])=2=n$.

A célula abaixo confere a classificação pelos postos (Teorema de Rouché–Capelli).

In [ ]:
import numpy as np

def classifica(A, b):
    # Classifica o sistema comparando posto(A), posto([A|b]) e o nº de incógnitas
    A = np.array(A, dtype=float)
    Ab = np.hstack([A, np.array(b, dtype=float).reshape(-1, 1)])
    rA, rAb, n = np.linalg.matrix_rank(A), np.linalg.matrix_rank(Ab), A.shape[1]
    if rA < rAb:
        return f"posto(A)={rA}, posto([A|b])={rAb} -> sem solução"
    if rA == n:
        return f"posto(A)={rA}, posto([A|b])={rAb} -> solução única"
    return f"posto(A)={rA}, posto([A|b])={rAb} -> infinitas soluções"

print("3a:", classifica([[1, 1], [2, 2]], [3, 6]))
print("3b:", classifica([[1, 1], [2, 2]], [3, 7]))
print("3c:", classifica([[1, 1], [2, -1]], [3, 0]))

3a: posto(A)=1, posto([A|b])=1 -> infinitas soluções
3b: posto(A)=1, posto([A|b])=2 -> sem solução
3c: posto(A)=2, posto([A|b])=2 -> solução única


# Parte 3 — Implementação numérica em Python

## Exercício 4 — Eliminação de Gauss sem pivoteamento

In [ ]:
import numpy as np

def gauss_sem_pivoteamento(A, b):
    # Resolve Ax = b por eliminação de Gauss SEM pivoteamento.
    # Retorna o vetor solução x.
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    n = A.shape[0]

    # 1) Matriz aumentada [A | b]
    Ab = np.hstack([A, b])

    # 2) Eliminação: zera os elementos abaixo de cada pivô
    for k in range(n - 1):
        if abs(Ab[k, k]) < 1e-12:
            # Sem pivoteamento, um pivô nulo inviabiliza o método
            raise ZeroDivisionError(f"Pivô nulo na linha {k}: seria necessário pivotear.")
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]                 # multiplicador m_ik
            Ab[i, k:] = Ab[i, k:] - m * Ab[k, k:]   # Li <- Li - m*Lk

    # 3) Substituição regressiva (de baixo para cima)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = Ab[i, i + 1:n] @ x[i + 1:]           # termos já conhecidos
        x[i] = (Ab[i, -1] - soma) / Ab[i, i]

    return x

Teste com o sistema do exercício 2 e comparação com `numpy.linalg.solve`:

In [ ]:
A2 = [[2, 1, -1],
      [-3, -1, 2],
      [-2, 1, 2]]
b2 = [8, -11, -3]

x_gauss = gauss_sem_pivoteamento(A2, b2)
x_numpy = np.linalg.solve(A2, b2)

print("Gauss  :", x_gauss)
print("NumPy  :", x_numpy)
print("Iguais?", np.allclose(x_gauss, x_numpy))
print("Resíduo ||Ax-b||:", np.linalg.norm(np.array(A2) @ x_gauss - b2))

Gauss  : [ 2.  3. -1.]
NumPy  : [ 2.  3. -1.]
Iguais? True
Resíduo ||Ax-b||: 0.0


**Interpretação:** os dois métodos coincidem até a precisão de máquina e a solução é $(2,3,-1)$, igual à resolução manual do exercício 2. Os pivôs foram $2$, $0{,}5$ e $-1$, todos não nulos, então o método sem pivoteamento funciona aqui. O `np.linalg.solve` usa fatoração LU com pivoteamento parcial, que é mais robusta: se um pivô fosse zero (ou muito pequeno), o nosso código falharia ou perderia precisão. Por isso a função lança um erro quando encontra pivô nulo.

# Parte 4 — Aplicação

## Exercício 5 — Modelo de balanço térmico

$$\begin{bmatrix}4&-1&0\\-1&4&-1\\0&-1&3\end{bmatrix}\begin{bmatrix}T_1\\T_2\\T_3\end{bmatrix}=\begin{bmatrix}15\\10\\10\end{bmatrix}$$

### a) Resolução manual

$$\left[\begin{array}{ccc|c}4&-1&0&15\\-1&4&-1&10\\0&-1&3&10\end{array}\right]$$

- $L_2\leftarrow L_2+\tfrac14 L_1$: $\;[\,0\;\;\tfrac{15}{4}\;\;-1\;|\;\tfrac{55}{4}\,]$
- $L_3\leftarrow L_3+\tfrac{4}{15}L_2$: $\;[\,0\;\;0\;\;\tfrac{41}{15}\;|\;\tfrac{41}{3}\,]$

**Substituição regressiva:**

- $\tfrac{41}{15}T_3=\tfrac{41}{3}\Rightarrow T_3=5$
- $\tfrac{15}{4}T_2-5=\tfrac{55}{4}\Rightarrow T_2=5$
- $4T_1-5=15\Rightarrow T_1=5$

$$\boxed{T_1=T_2=T_3=5}$$

Verificação: $4(5)-5=15$ ✓; $-5+20-5=10$ ✓; $-5+3(5)=10$ ✓.

### b) Implementação em Python

In [ ]:
A5 = [[4, -1, 0],
      [-1, 4, -1],
      [0, -1, 3]]
b5 = [15, 10, 10]

T_gauss = gauss_sem_pivoteamento(A5, b5)
T_numpy = np.linalg.solve(A5, b5)

print("T (Gauss):", T_gauss)
print("T (NumPy):", T_numpy)
print("Iguais?", np.allclose(T_gauss, T_numpy))

T (Gauss): [5. 5. 5.]
T (NumPy): [5. 5. 5.]
Iguais? True


**Interpretação física e numérica:** as três temperaturas são iguais a 5 e o código confirma a resolução manual. A matriz é *diagonalmente dominante* ($|4|>|-1|$, $|4|>|-1|+|-1|$, $|3|>|-1|$), o que garante pivôs não nulos e bom comportamento numérico, por isso o método sem pivoteamento funciona bem.